In [ ]:
import pandas as pd
import json
import os
path_file = '/kaggle/input/3tand1k/DATASET/12.1.2026.listens'
output_file = '/kaggle/working/cleaned_music_data_12_1.csv'

# cột gốc cần giữ
cols_to_keep = [
    'user_id', 
    'user_name', 
    'timestamp', 
    'recording_msid',
    'track_metadata.track_name', 
    'track_metadata.artist_name', 
    'track_metadata.release_name',
    'track_metadata.additional_info.submission_client',
    'track_metadata.additional_info.recording_msid',
    'track_metadata.additional_info.lastfm_artist_mbid'
]

# đổi tên cột 
rename_map = {
    'track_metadata.track_name': 'track_name',
    'track_metadata.artist_name': 'artist_name',
    'track_metadata.release_name': 'release_name',
    'track_metadata.additional_info.submission_client': 'client_app',
    'track_metadata.additional_info.recording_msid': 'additional_msid',
    'track_metadata.additional_info.lastfm_artist_mbid': 'lastfm_mbid'
}

data = []
with open(path_file, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            try:
                data.append(json.loads(line))
            except json.JSONDecodeError:
                continue 

# tạo DataFrame và normalize dữ liệu 
df = pd.json_normalize(data)

# lọc cột và đổi tên
existing_cols = [c for c in cols_to_keep if c in df.columns]
df_clean = df[existing_cols].copy()

# Đổi tên cột
df_clean.rename(columns=rename_map, inplace=True)

# làm sạch
# chuyển timestamp sang dạng datetime cho dễ đọc
if 'timestamp' in df_clean.columns:
    df_clean['timestamp'] = pd.to_datetime(df_clean['timestamp'], unit='s', errors='coerce')

str_cols = df_clean.select_dtypes(include=['object']).columns
for col in str_cols:
    df_clean[col] = df_clean[col].str.strip()
df_clean.drop_duplicates(inplace=True)

# lưu file
print(f"Đang lưu file xuống: {output_file}")
df_clean.to_csv(output_file, index=False, encoding='utf-8')
print("Hoàn tất!")
